# 01 · JSON e CRUD

**Pré-requisito:** notebook 00 concluído (MongoDB rodando e arquivo `.env` criado).

**Nesta etapa:** JSON em Python, conexão com o MongoDB e as quatro operações básicas (CRUD). No fim, o **Exercício 1**.

## Conexão com o seu MongoDB

A célula abaixo lê o endereço do banco (`MONGO_URI`) do arquivo **`.env`** da pasta, criado no notebook 00, e testa a conexão. Se der erro, confira se o MongoDB está rodando (notebook 00, Etapa 4).

In [31]:
import os
from dotenv import load_dotenv
from pymongo import MongoClient

load_dotenv()                                   # lê o arquivo .env da pasta
client = MongoClient(os.environ["MONGO_URI"])
client.admin.command("ping")
print("Conectado ao MongoDB")

Conectado ao MongoDB


---
## 1. JSON em Python

Um **objeto JSON** é um conjunto de pares `"chave": valor` entre chaves `{}`, como uma ficha cadastral. Em Python, ele corresponde a um **dicionário** (`dict`).

In [32]:
professor = {
    "nome": "Rafael",
    "idade": 38,
    "profissao": "Professor",
    "temFilhos": True,
}
professor

{'nome': 'Rafael', 'idade': 38, 'profissao': 'Professor', 'temFilhos': True}

Objetos podem conter listas e outros objetos (fichas dentro de fichas):

In [ ]:
aluno = {
    "nome": "Ana Souza",
    "idade": 21,
    "curso": "Ciência de Dados",
    "notas": [8.5, 9.0, 7.8],
    "endereco": {"cidade": "São Paulo", "uf": "SP"},
    "ativo": True,
}


São Paulo
Média: 8.433333333333334


In [ ]:
print(aluno["endereco"]["cidade"])
print("Média:", sum(aluno["notas"]) / len(aluno["notas"]))

### Serializar e desserializar

Para **viajar** (pela rede, para um arquivo, para outro programa), o objeto vira **texto JSON**. Do outro lado, o texto vira objeto de novo.

In [34]:
import json

texto = json.dumps(aluno, ensure_ascii=False)   # empacotar: dict → texto
print(type(texto), texto)

de_volta = json.loads(texto)                     # desempacotar: texto → dict
print(type(de_volta), de_volta["nome"])

<class 'str'> {"nome": "Ana Souza", "idade": 21, "curso": "Ciência de Dados", "notas": [8.5, 9.0, 7.8], "endereco": {"cidade": "São Paulo", "uf": "SP"}, "ativo": true}
<class 'dict'> Ana Souza


### JSON de verdade: uma API pública

Quase toda API web responde em JSON. A ViaCEP, por exemplo, devolve o endereço de um CEP:

In [35]:
import requests

try:
    endereco = requests.get("https://viacep.com.br/ws/01001000/json/", timeout=10).json()   # JSON → dict
    print(endereco)
except Exception as e:
    print("Sem acesso à API agora:", e)
    endereco = {"cep": "01001-000", "logradouro": "Praça da Sé", "localidade": "São Paulo", "uf": "SP"}

{'cep': '01001-000', 'logradouro': 'Praça da Sé', 'complemento': 'lado ímpar', 'unidade': '', 'bairro': 'Sé', 'localidade': 'São Paulo', 'uf': 'SP', 'estado': 'São Paulo', 'regiao': 'Sudeste', 'ibge': '3550308', 'gia': '1004', 'ddd': '11', 'siafi': '7107'}


Repare: o que chegou da API já é um `dict`, e um `dict` é exatamente o que o MongoDB guarda. **Não há tradução no meio.**

## 2. Banco, coleção e documento

| SQL | MongoDB |
|---|---|
| banco de dados | banco de dados |
| tabela | **coleção** |
| linha | **documento** |
| coluna | campo |
| chave primária | `_id` |

In [36]:
db = client["escola"]           # banco
alunos = db["alunos"]           # coleção

db["enderecos"].insert_one(endereco)    # o JSON da API vai direto para o banco
print("Endereços guardados:", db["enderecos"].count_documents({}))

Endereços guardados: 2


## 3. CRUD

### C · Create (inserir)

In [37]:
ALUNOS = [
    {"nome": "Ana",     "idade": 22, "curso": "Computação"},
    {"nome": "Bruno",   "idade": 19, "curso": "Medicina"},
    {"nome": "Carla",   "idade": 24, "curso": "Direito"},
    {"nome": "Matheus", "idade": 21, "curso": "Computação", "disciplinas": ["Python"]},
    {"nome": "Mariana", "idade": 26, "curso": "Engenharia"},
    {"nome": "Pedro",   "idade": 20, "curso": "Medicina"},
    {"nome": "Juliana", "idade": 23, "curso": "Ciência de Dados"},
    {"nome": "Lucas",   "idade": 18, "curso": "Direito"},
]

alunos.delete_many({})                        # começa do zero a cada execução
resultado = alunos.insert_many(ALUNOS)
print(len(resultado.inserted_ids), "alunos inseridos")

8 alunos inseridos


Só o Matheus tem `disciplinas`, e ninguém reclamou: cada documento pode ter campos diferentes.

Cada documento ganhou um `_id` do tipo **ObjectId** (12 bytes: data de criação + valor aleatório + contador).

In [38]:
doc = alunos.find_one({"nome": "Ana"})
print(doc["_id"])
print("Criado em:", doc["_id"].generation_time)

6b035ddb912cc6369ee77aca
Criado em: 2026-11-22 21:25:47+00:00


### R · Read (ler)

- `find_one(filtro)` devolve **um** documento (ou `None`).
- `find(filtro, projeção)` devolve um **cursor**, percorrido com `for`.


In [39]:
for a in alunos.find({"curso": "Medicina"}):
    print(a["nome"], a["idade"])

Bruno 19
Pedro 20


- Na projeção, `1` mostra o campo e `0` esconde. O `_id` aparece sempre, a menos que você diga `0`.

In [40]:
for a in alunos.find({}, {"_id": 0, "nome": 1, "curso": 1}).sort("idade", -1):
    print(a)

{'nome': 'Mariana', 'curso': 'Engenharia'}
{'nome': 'Carla', 'curso': 'Direito'}
{'nome': 'Juliana', 'curso': 'Ciência de Dados'}
{'nome': 'Ana', 'curso': 'Computação'}
{'nome': 'Matheus', 'curso': 'Computação'}
{'nome': 'Pedro', 'curso': 'Medicina'}
{'nome': 'Bruno', 'curso': 'Medicina'}
{'nome': 'Lucas', 'curso': 'Direito'}


In [41]:
print("Total:", alunos.count_documents({}))
print("De Direito:", alunos.count_documents({"curso": "Direito"}))

Total: 8
De Direito: 2


### U · Update (alterar)

`update_one(filtro, alteração)` e `update_many(...)`. A alteração usa operadores como **`$set`**.

In [50]:
alunos.update_one({"nome": "Ana"}, {"$set": {"idade": 23}})
print(alunos.find_one({"nome": "Ana"}, {"_id": 0}))

{'nome': 'Ana', 'idade': 23, 'curso': 'Computação'}


In [51]:
alunos.update_many({"curso": "Direito"}, {"$set": {"campus": "Centro"}})
print(alunos.find_one({"nome": "Carla"}, {"_id": 0}))

{'nome': 'Carla', 'idade': 24, 'curso': 'Direito', 'campus': 'Centro'}


> **Atenção:** `replace_one(filtro, novo)` troca o documento **inteiro**: todos os campos que você não repetir somem. Para mudar só alguns campos, use `update_one` com `$set`.

In [52]:
print("Antes:          ", alunos.find_one({"nome": "Bruno"}, {"_id": 0}))

# update_one + $set: muda só a idade, o resto fica
alunos.update_one({"nome": "Bruno"}, {"$set": {"idade": 20}})
print("Com $set:       ", alunos.find_one({"nome": "Bruno"}, {"_id": 0}))

# replace_one: o documento vira exatamente o que foi passado
alunos.replace_one({"nome": "Bruno"}, {"nome": "Bruno", "idade": 20})
print("Com replace_one:", alunos.find_one({"nome": "Bruno"}, {"_id": 0}))

Antes:           {'nome': 'Bruno', 'idade': 19, 'curso': 'Medicina'}
Com $set:        {'nome': 'Bruno', 'idade': 20, 'curso': 'Medicina'}
Com replace_one: {'nome': 'Bruno', 'idade': 20}


In [53]:
alunos.delete_one({"nome": "Lucas"})
print("Depois de apagar o Lucas:", alunos.count_documents({}))

# alunos.delete_many({})   # cuidado: apagaria TODOS os documentos da coleção

Depois de apagar o Lucas: 7


## Agora vamos fazer o Exercício 1 · Sua primeira coleção

**Você vai usar:** `insert_many`, `insert_one`, `find` com projeção, `update_one` + `$set`, `count_documents`.

1. Garanta que `escola.alunos` tem os **8 alunos** da lista `ALUNOS` (o Lucas foi apagado acima: recrie a coleção).
2. Insira **você mesmo**, com um objeto aninhado `endereco` (cidade e UF) e uma lista `notas` com três notas.
3. Mostre apenas `nome` e `curso` de todos os alunos, **sem** o `_id`.
4. Acrescente o campo `ativo: True` ao **seu** documento e leia-o de volta.
5. Conte quantos alunos existem no curso de **Computação**.

In [44]:
# 1.1 Recriar a coleção com os 8 alunos
# seu código aqui

In [45]:
# 1.2 Inserir você mesmo (com endereco e notas)
# seu código aqui

In [46]:
# 1.3 Mostrar nome e curso, sem _id
# seu código aqui

In [47]:
# 1.4 Acrescentar ativo: True ao seu documento
# seu código aqui

In [48]:
# 1.5 Contar alunos de Computação
# seu código aqui

### Confira o seu Exercício 1

In [49]:
total = alunos.count_documents({})
com_endereco = alunos.count_documents({"endereco.cidade": {"$exists": True}, "notas": {"$size": 3}})
ativos = alunos.count_documents({"ativo": True})

print("9 alunos na coleção:          ", "OK" if total == 9 else f"FALTA (tem {total})")
print("1 aluno com endereco e 3 notas:", "OK" if com_endereco >= 1 else "FALTA")
print("1 aluno com ativo: True:       ", "OK" if ativos >= 1 else "FALTA")

9 alunos na coleção:           FALTA (tem 7)
1 aluno com endereco e 3 notas: FALTA
1 aluno com ativo: True:        FALTA


**Próximo:** `02_Consultas_e_Agregacao.ipynb`